In [1]:
from bloqade.analog.atom_arrangement import Honeycomb, Square, Chain, Kagome

# 使用预定义晶格
# Honeycomb(行数, 列数, lattice_spacing=间距)
geometry = Honeycomb(2, lattice_spacing=10.0)

# 其他晶格示例
square_geom = Square(3, lattice_spacing=6.5)
chain_geom = Chain(5)
kagome_geom = Kagome(3)

# 自定义位置
from bloqade.analog import start
custom_geom = start.add_position([(0, 0), (6, 0), (12, 0)])

# 添加缺陷
defective = Square(4, lattice_spacing=5.0).apply_defect_density(0.2)

# 可视化
defective.show()
geometry.show()

C:\Users\26676\anaconda3\envs\ml\lib\site-packages\bloqade\analog\__init__.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)


In [2]:
# python后端 + 波形扫描 + 阻塞验证（失败） 
# Ω 必须满足：足够大，使得完成至少一个 Rabi 周期 (Ω > 2π/T_sweep)
# Ω 足够小，使得非共振泄漏被抑制 (Ω²/V ≪ dΔ/dt)
from math import pi, sin
import matplotlib.pyplot as plt

from bloqade.analog import start
from bloqade.analog.atom_arrangement import Chain


# ======================================================
# Rydberg 物理参数 — ⁸⁷Rb, |70S₁/₂⟩
# ======================================================

C6 = 862690 * 2 * pi               # ≈ 5.42 × 10⁶ rad·μm⁶/μs

# ① 原子间距
spacing = 6.0                       # μm

# ② vdW 在 spacing 处的强度
V_at_spacing = C6 / (spacing ** 6)  # ≈ 1323 rad/μs
print(f"V(spacing={spacing}) = {V_at_spacing:.0f} rad/μs")

# ③ 目标 V/Ω 比值
V_over_Omega_target = 25            # V > 25×Ω → 强阻塞

# ④ 反推 Ω_max
omega_max = V_at_spacing / V_over_Omega_target  # ≈ 53 rad/μs
print(f"Ω_max = {omega_max:.1f} rad/μs")

# ⑤ 由 Ω_max 计算 Rb（结果，不是输入）
Rb = (C6 / omega_max) ** (1.0 / 6.0)
print(f"R_b  = {Rb:.2f} μm")
print(f"验证: V/Ω × Ω_max = {V_at_spacing/omega_max:.1f}×")
print(f"       spacing < R_b? {spacing < Rb}  → 近邻 vdW 被计算")
print(f"       2×spacing < R_b? {2*spacing < Rb}  → 次近邻 vdW {'被计算' if 2*spacing < Rb else '不算'}（{'好——次近邻不应阻塞' if 2*spacing >= Rb else '注意——次近邻也被截断'})\n")


# ======================================================
# 1. 原子排列
# ======================================================

geometry = Chain(3, lattice_spacing=spacing)


# ======================================================
# 2. Rabi 波形 — 统一用 Ω_max
# ======================================================

Ω = omega_max

program_linear = (
    geometry.rydberg.rabi.amplitude.uniform
    .piecewise_linear(durations=[1.0, 2.0, 1.0], values=[0, Ω, Ω, 0])
)

program_constant = (
    geometry.rydberg.rabi.amplitude.uniform
    .constant(value=Ω, duration=3.0)
)

program_piecewise = (
    geometry.rydberg.rabi.amplitude.uniform
    .piecewise_constant(durations=[1.0, 1.0, 1.0], values=[0.4*Ω, Ω, 0.4*Ω])
)

scale = Ω / 6.25
program_poly = (
    geometry.rydberg.rabi.amplitude.uniform
    .poly(coeffs=[0, 5*scale, -1*scale], duration=3.0)
)

def custom_wave(t):
    return Ω * abs(sin(t))

program_custom = (
    geometry.rydberg.rabi.amplitude.uniform
    .fn(custom_wave, duration=3.0)
)


# ======================================================
# 3. Detuning 扫频
# ======================================================

def add_detuning(program):
    return (
        program.detuning.uniform
        .piecewise_linear(
            durations=[1.0, 2.0, 1.0],
            values=[-10, -10, 10, 10]
        )
    )


programs = {
    "linear":    add_detuning(program_linear),
    "constant":  add_detuning(program_constant),
    "piecewise": add_detuning(program_piecewise),
    "poly":      add_detuning(program_poly),
    "custom":    add_detuning(program_custom),
}


# ======================================================
# 4. 模拟
# ======================================================

results = {}

for name, program in programs.items():
    print(f"===================\n{name}\n===================")

    result = program.bloqade.python().run(
        shots=1000,
        blockade_radius=Rb   
    )

    results[name] = result
    report = result.report()
    counts = report.counts()
    print("Bitstring counts:", counts)
    print()


V(spacing=6.0) = 116 rad/μs
Ω_max = 4.6 rad/μs
R_b  = 10.26 μm
验证: V/Ω × Ω_max = 25.0×
       spacing < R_b? True  → 近邻 vdW 被计算
       2×spacing < R_b? False  → 次近邻 vdW 不算（好——次近邻不应阻塞)

linear
Bitstring counts: [OrderedDict([('010', 869), ('101', 101), ('011', 18), ('110', 11), ('111', 1)])]

constant
Bitstring counts: [OrderedDict([('010', 779), ('111', 97), ('101', 56), ('110', 42), ('011', 26)])]

piecewise
Bitstring counts: [OrderedDict([('010', 466), ('101', 288), ('011', 100), ('111', 74), ('110', 72)])]

poly
Bitstring counts: [OrderedDict([('010', 805), ('101', 90), ('011', 51), ('110', 46), ('111', 8)])]

custom
Bitstring counts: [OrderedDict([('010', 739), ('101', 173), ('011', 43), ('110', 43), ('111', 2)])]



In [3]:
# ======================================================
# Braket 后端 vs Python 后端
# ======================================================
# 任意波形发生器 (AWG)  的输出电压不能瞬时跳变
# 最好输入piecewise-linear
# Python 后端 (.bloqade.python()):
#   - 用 blockade_radius 做硬阻塞近似
#   - 距离 < Rb:  计算 vdW (V = C₆/r⁶)
#   - 距离 ≥ Rb:  V = 0（完全不计算）
#   - 快，但 vdW 是阶跃函数，不是平滑物理
#
# Braket 后端 (.braket.local_emulator()):
#   - 自动对所有原子对计算平滑的 C₆/r⁶ vdW 相互作用
#   - 不需要 blockade_radius 参数
#   - 更物理，但大系统时较慢
# ======================================================
#
# Ω 必须满足：足够大，使得完成至少一个 Rabi 周期 (Ω > 2π/T_sweep)
# Ω 足够小，使得非共振泄漏被抑制 (Ω²/V ≪ dΔ/dt)
# ======================================================

from math import pi, sin
import matplotlib.pyplot as plt

from bloqade.analog import start
from bloqade.analog.atom_arrangement import Chain


# ======================================================
# Rydberg 物理参数 — ⁸⁷Rb, |70S₁/₂⟩
# ======================================================

C6 = 862690 * 2 * pi               # ≈ 5.42 × 10⁶ rad·μm⁶/μs

# ① 原子间距
spacing = 6.0                       # μm

# ② vdW 在 spacing 处的强度
V_at_spacing = C6 / (spacing ** 6)
print(f"V(spacing={spacing}) = {V_at_spacing:.0f} rad/μs")

# ③ 目标 V/Ω 比值
V_over_Omega_target = 25

# ④ 反推 Ω_max
omega_max = V_at_spacing / V_over_Omega_target
print(f"Ω_max = {omega_max:.1f} rad/μs")
print(f"Rabi 周期 = {2*pi/omega_max:.2f} μs  (平台 2μs ≈ {2/(2*pi/omega_max):.1f} 个周期)")
print(f"验证: V/Ω = {V_at_spacing/omega_max:.1f}×")
print(f"       Ω²/V = {omega_max**2/V_at_spacing:.2f}  (应 ≪ dΔ/dt ≈ 10)\n")

# Braket 后端不需要 blockade_radius，但仍计算 Rb 做参考
Rb_physical = (C6 / omega_max) ** (1.0 / 6.0)
print(f"R_b (物理参考值) = {Rb_physical:.2f} μm")
print(f"spacing < R_b? {spacing < Rb_physical}  → 近邻应被阻塞\n")


# ======================================================
# 1. 原子排列
# ======================================================

geometry = Chain(3, lattice_spacing=spacing)


# ======================================================
# 2. Rabi 波形 — 统一用 Ω_max
# ======================================================

Ω = omega_max

# Braket 要求波形首尾连续到 0，且 Rabi 与 detuning 时长必须一致 (全部 4.0 μs)

program_linear = (
    geometry.rydberg.rabi.amplitude.uniform
    .piecewise_linear(durations=[1.0, 2.0, 1.0], values=[0, Ω, Ω, 0])
)

program_constant = (
    geometry.rydberg.rabi.amplitude.uniform
    .piecewise_linear(durations=[0.05, 3.9, 0.05], values=[0, Ω, Ω, 0])
)



# ======================================================
# 3. Detuning 扫频
# ======================================================

def add_detuning(program):
    return (
        program.detuning.uniform
        .piecewise_linear(
            durations=[1.0, 2.0, 1.0],
            values=[-10, -10, 10, 10]
        )
    )


programs = {
    "linear":    add_detuning(program_linear),
    "constant":  add_detuning(program_constant),
}


# ======================================================
# 4. 模拟 — Braket 后端（完整 C₆/r⁶ vdW）
# ======================================================

results = {}

for name, program in programs.items():
    print(f"===================\n{name}\n===================")

    result = program.braket.local_emulator().run(shots=1000)
    # Braket 后端自动对所有原子对计算 V_ij = C₆/r_ij⁶
    # 无需 blockade_radius 参数

    results[name] = result
    report = result.report()
    counts = report.counts()
    print("Bitstring counts:", counts)
    print()


V(spacing=6.0) = 116 rad/μs
Ω_max = 4.6 rad/μs
Rabi 周期 = 1.35 μs  (平台 2μs ≈ 1.5 个周期)
验证: V/Ω = 25.0×
       Ω²/V = 0.19  (应 ≪ dΔ/dt ≈ 10)

R_b (物理参考值) = 10.26 μm
spacing < R_b? True  → 近邻应被阻塞

linear
Bitstring counts: [OrderedDict([('010', 849), ('101', 113), ('011', 20), ('110', 16), ('111', 2)])]

constant
Bitstring counts: [OrderedDict([('010', 457), ('110', 207), ('011', 199), ('111', 93), ('101', 44)])]



In [3]:
# 边着色技术对 QAOA 电路做编译优化
import math
import networkx as nx
from kirin.dialects import ilist
from bloqade import qasm2
from typing import Any

pi = math.pi
N = 32
G = nx.random_regular_graph(3, N, seed=42)

# 通过边着色实现 SIMD 并行化
def qaoa_simd(G: nx.Graph):
    nodes = list(G.nodes)
    Gline = nx.line_graph(G)
    colors = nx.algorithms.coloring.equitable_color(Gline, num_colors=5)
    left_ids = ilist.IList([ilist.IList([edge[0] for edge in G.edges if colors[edge] == i]) for i in range(5)])
    right_ids = ilist.IList([ilist.IList([edge[1] for edge in G.edges if colors[edge] == i]) for i in range(5)])

    @qasm2.extended
    def parallel_h(qargs):
        qasm2.parallel.u(qargs=qargs, theta=pi/2, phi=0.0, lam=pi)

    @qasm2.extended
    def parallel_cx(ctrls, qargs):
        parallel_h(qargs)
        qasm2.parallel.cz(ctrls, qargs)
        parallel_h(qargs)

    @qasm2.extended
    def parallel_cz_phase(ctrls, qargs, gamma):
        parallel_cx(ctrls, qargs)
        qasm2.parallel.rz(qargs, gamma)
        parallel_cx(ctrls, qargs)

    @qasm2.extended
    def kernel(gamma: ilist.IList[float, Any], beta: ilist.IList[float, Any]):
        q = qasm2.qreg(len(nodes))
        def get_qubit(x): return q[x]
        all_qubits = ilist.map(fn=get_qubit, collection=range(N))
        parallel_h(all_qubits)

        for i in range(len(gamma)):
            for cind in range(5):
                ctrls = ilist.map(fn=get_qubit, collection=left_ids[cind])
                qargs = ilist.map(fn=get_qubit, collection=right_ids[cind])
                parallel_cz_phase(ctrls, qargs, gamma[i])
            qasm2.parallel.u(qargs=all_qubits, theta=beta[i], phi=0.0, lam=0.0)
        return q
    return kernel

# 编译并运行
kernel = qaoa_simd(G)
@qasm2.extended
def main():
    kernel([0.1, 0.2], [0.3, 0.4])

target = qasm2.emit.QASM2()
ast = target.emit(main)
qasm2.parse.pprint(ast)

OPENQASM 2.0;
include "qelib1.inc";
qreg q[32];
U(1.5707963267948966, 0.0, 3.141592653589793) q[31];
U(1.5707963267948966, 0.0, 3.141592653589793) q[30];
U(1.5707963267948966, 0.0, 3.141592653589793) q[29];
U(1.5707963267948966, 0.0, 3.141592653589793) q[28];
U(1.5707963267948966, 0.0, 3.141592653589793) q[27];
U(1.5707963267948966, 0.0, 3.141592653589793) q[26];
U(1.5707963267948966, 0.0, 3.141592653589793) q[25];
U(1.5707963267948966, 0.0, 3.141592653589793) q[24];
U(1.5707963267948966, 0.0, 3.141592653589793) q[23];
U(1.5707963267948966, 0.0, 3.141592653589793) q[22];
U(1.5707963267948966, 0.0, 3.141592653589793) q[21];
U(1.5707963267948966, 0.0, 3.141592653589793) q[20];
U(1.5707963267948966, 0.0, 3.141592653589793) q[19];
U(1.5707963267948966, 0.0, 3.141592653589793) q[18];
U(1.5707963267948966, 0.0, 3.141592653589793) q[17];
U(1.5707963267948966, 0.0, 3.141592653589793) q[16];
U(1.5707963267948966, 0.0, 3.141592653589793) q[15];
U(1.5707963267948966, 0.0, 3.141592653589793) q[14]

In [4]:
import numpy as np
from scipy.optimize import minimize
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
import networkx as nx

# =========================
# Problem
# =========================
N = 8
G = nx.random_regular_graph(d=3, n=N, seed=42)
edges = list(G.edges)

print(f"Graph: {N} nodes, {len(edges)} edges (3-regular)")

def calculate_cost(bitstring):
    """MaxCut: count edges crossing the partition."""
    cost = 0
    for i, j in edges:
        if bitstring[N - 1 - i] != bitstring[N - 1 - j]:
            cost += 1
    return cost


# =========================
# QAOA circuit
# =========================
def qaoa_circuit(gamma, beta, p):
    qc = QuantumCircuit(N, N)

    # |+>^⊗N
    for i in range(N):
        qc.h(i)

    # p layers
    for layer in range(p):
        # ── Cost: e^{-iγ H_C} ──
        for i, j in edges:
            qc.cx(i, j)
            qc.rz(2.0 * gamma[layer], j)
            qc.cx(i, j)

        # ── Mixer: e^{-iβ H_M} ──
        for i in range(N):
            qc.rx(2.0 * beta[layer], i)

    # ── Measurement (once, after all layers) ──
    qc.measure(range(N), range(N))
    return qc


# =========================
# Execution
# =========================
simulator = AerSimulator()


def run_qaoa(gamma, beta, p, shots=1000):
    qc = qaoa_circuit(gamma, beta, p)
    result = simulator.run(qc, shots=shots).result()
    return result.get_counts()


# =========================
# Objective: maximize cut -> minimize -cut
# =========================
p = 2


def objective(params):
    gamma = params[:p]
    beta = params[p:]

    counts = run_qaoa(gamma, beta, p, shots=1000)
    total = sum(counts.values())

    energy = 0.0
    for bitstring, freq in counts.items():
        energy += calculate_cost(bitstring) * freq

    energy /= total
    return -energy


# =========================
# Optimization
# =========================
initial_params = np.random.uniform(0, np.pi, 2 * p)

result = minimize(objective, initial_params, method="COBYLA",
                  options={"maxiter": 100})

gamma_opt = result.x[:p]
beta_opt = result.x[p:]

print("=" * 30)
print("Optimization done")
print(f"gamma = {gamma_opt}")
print(f"beta  = {beta_opt}")

# =========================
# Final sampling
# =========================
counts = run_qaoa(gamma_opt, beta_opt, p, shots=5000)

best = max(counts, key=calculate_cost)
print(f"\nBest solution : {best}")
print(f"MaxCut value  : {calculate_cost(best)}")

print(f"\nTop 5 bitstrings:")
for bs in sorted(counts, key=lambda x: counts[x], reverse=True)[:5]:
    print(f"  {bs} : {counts[bs]:5d}  cut={calculate_cost(bs)}")


Graph: 8 nodes, 12 edges (3-regular)
Optimization done
gamma = [2.02468613 2.94533493]
beta  = [-0.37318245  3.8481933 ]

Best solution : 01100110
MaxCut value  : 10

Top 5 bitstrings:
  01101010 :   391  cut=10
  01100110 :   387  cut=10
  10010101 :   381  cut=10
  10011001 :   364  cut=10
  00011001 :   147  cut=9


In [8]:
import numpy as np
from scipy.optimize import minimize
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
import networkx as nx

# =========================
# Problem
# =========================
N = 24
G = nx.random_regular_graph(d=3, n=N, seed=42)
edges = list(G.edges)

print(f"Graph: {N} nodes, {len(edges)} edges (3-regular)")

def calculate_cost(bitstring):
    """MaxCut: count edges crossing the partition."""
    cost = 0
    for i, j in edges:
        if bitstring[N - 1 - i] != bitstring[N - 1 - j]:
            cost += 1
    return cost


# =========================
# QAOA circuit
# =========================
def qaoa_circuit(gamma, beta, p):
    qc = QuantumCircuit(N, N)

    # |+>^⊗N
    for i in range(N):
        qc.h(i)

    # p layers
    for layer in range(p):
        # ── Cost: e^{-iγ H_C} ──
        for i, j in edges:
            qc.cx(i, j)
            qc.rz(2.0 * gamma[layer], j)
            qc.cx(i, j)

        # ── Mixer: e^{-iβ H_M} ──
        for i in range(N):
            qc.rx(2.0 * beta[layer], i)

    # ── Measurement (once, after all layers) ──
    qc.measure(range(N), range(N))
    return qc


# =========================
# Execution
# =========================
simulator = AerSimulator()


def run_qaoa(gamma, beta, p, shots=1000):
    qc = qaoa_circuit(gamma, beta, p)
    result = simulator.run(qc, shots=shots).result()
    return result.get_counts()


# =========================
# Objective: maximize cut -> minimize -cut
# =========================
p = 2


def objective(params):
    gamma = params[:p]
    beta = params[p:]

    counts = run_qaoa(gamma, beta, p, shots=1000)
    total = sum(counts.values())

    energy = 0.0
    for bitstring, freq in counts.items():
        energy += calculate_cost(bitstring) * freq

    energy /= total
    return -energy


# =========================
# Optimization
# =========================
initial_params = np.random.uniform(0, np.pi, 2 * p)

result = minimize(objective, initial_params, method="COBYLA",
                  options={"maxiter": 100})

gamma_opt = result.x[:p]
beta_opt = result.x[p:]

print("=" * 30)
print("Optimization done")
print(f"gamma = {gamma_opt}")
print(f"beta  = {beta_opt}")

# =========================
# Final sampling
# =========================
counts = run_qaoa(gamma_opt, beta_opt, p, shots=5000)

best = max(counts, key=calculate_cost)
print(f"\nBest solution : {best}")
print(f"MaxCut value  : {calculate_cost(best)}")

print(f"\nTop 5 bitstrings:")
for bs in sorted(counts, key=lambda x: counts[x], reverse=True)[:5]:
    print(f"  {bs} : {counts[bs]:5d}  cut={calculate_cost(bs)}")


Graph: 24 nodes, 36 edges (3-regular)
Optimization done
gamma = [-0.35404411  1.41984643]
beta  = [3.47632528 1.64702444]

Best solution : 001100110101010111010011
MaxCut value  : 31

Top 5 bitstrings:
  000110001110010110101001 :     2  cut=23
  011111100010001110101010 :     2  cut=27
  111001101100101010010010 :     2  cut=26
  101100011100010011010011 :     2  cut=28
  100100110101010101001110 :     2  cut=26


In [ ]:
"""
classic mathods:
new best: 32 at 000110111001011010101100
"""